In [ ]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
from transformers import AutoTokenizer
from pricer.items import Item

In [ ]:
LITE_MODE=True
from huggingface_hub import login
load_dotenv(override=True)
hf_token = os.environ['HF_token']
login(hf_token, add_to_git_credential=True)


In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv(override=True)
hf_token = os.environ['HF_token']
login(hf_token, add_to_git_credential=True)

# Then retry loading the model
from transformers import AutoTokenizer
BASE_MODEL = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL,token=hf_token)

In [ ]:
token_counts = [item.countTokens(tokenizer) for item in tqdm(train)]


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))
plt.title(f"Tokens in Summary: Avg {sum(token_counts)/len(token_counts):,.1f} and highest {max(token_counts):,}\n")
plt.xlabel('Number of tokens in summary')
plt.ylabel('Count')
plt.hist(token_counts, rwidth=0.7, color="skyblue", bins=range(0, 200, 10))
plt.show()

In [ ]:
CUTOFF = 110
for item in tqdm(train+val):
    item.make_prompts(tokenizer, CUTOFF, True)
for item in tqdm(test):
    item.make_prompts(tokenizer, CUTOFF, False)

In [ ]:
print("PROMPT:")
print(test[0].prompt)
print("COMPLETION:")
print(test[0].completion)


In [ ]:
prompt_token_counts = [item.count_prompt_tokens(tokenizer) for item in tqdm(items)]

In [ ]:
len(prompt_token_counts)


In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Tokens: Avg {sum(prompt_token_counts)/len(prompt_token_counts):,.1f} and highest {max(prompt_token_counts):,}\n")
plt.xlabel('Number of tokens in prompt and the completion')
plt.ylabel('Count')
plt.hist(prompt_token_counts, rwidth=0.7, color="gold", bins=range(0, 200, 10))
plt.show()